In [ ]:
import os
import time
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import TFViTForImageClassification
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from pathlib import Path
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,s
)

from transformers import TFViTForImageClassification

# ============================================================
# CONFIGURATION
# ============================================================
MODEL_NAME = "ViT"
OUTPUT_DIR = MODEL_NAME

DATASET_PATH = r"../DATASETS/Rice_Leaf_AUG"   # <-- adjust to wherever DATASETS/ sits relative to your notebook
IMG_SIZE = 224
BATCH_SIZE = 16          # kept from the original ViT script (ViT is heavier than MobileNetV2)
EPOCHS = 1
SEED = 123
LEARNING_RATE = 5e-5     # kept from the original ViT script
HF_CHECKPOINT = "google/vit-base-patch16-224"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def outp(*parts):
    return os.path.join(OUTPUT_DIR, *parts)


# ============================================================
# LOAD DATASET (STRATIFIED) -- identical logic to the MobileNetV2 script
# ============================================================
dataset_path = Path(DATASET_PATH)

class_names = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

image_paths = []
labels = []

for cls in class_names:
    folder = dataset_path / cls
    for img in folder.iterdir():
        if img.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
            image_paths.append(str(img))
            labels.append(class_to_idx[cls])

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=SEED,
    shuffle=True,
)

print("Training images :", len(train_paths))
print("Validation images:", len(val_paths))

for i, cls in enumerate(class_names):
    print(cls)
    print(
        " Train:",
        sum(np.array(train_labels) == i),
        " Validation:",
        sum(np.array(val_labels) == i),
    )

AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    """
    ViT (google/vit-base-patch16-224) expects:
      - pixel values normalized to roughly [-1, 1] (mean=std=0.5)
      - channel-first layout: (C, H, W), not (H, W, C)
    """
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - 0.5) / 0.5
    image = tf.transpose(image, perm=[2, 0, 1])          # HWC -> CHW
    image.set_shape((3, IMG_SIZE, IMG_SIZE))
    return image, label


train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .shuffle(len(train_paths), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

validation_dataset = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

train_dataset_prefetched = train_dataset
validation_dataset_prefetched = validation_dataset

class_indices = list(range(len(class_names)))

# ============================================================
# MODEL: HuggingFace TF-ViT wrapped as a tf.keras.Model
# ============================================================
# NOTE: The original PyTorch script used no augmentation (just Resize + ToTensor),
# so none is added here either. If you want parity with the MobileNetV2 pipeline's
# augmentation, add a tf.keras.Sequential([...]) block and apply it to `inputs`
# below (remember it must run in HWC before the CHW transpose, or be redefined
# to operate on CHW tensors).

hf_vit = TFViTForImageClassification.from_pretrained(
    HF_CHECKPOINT,
    num_labels=len(class_names),
    ignore_mismatched_sizes=True,
)


class ViTClassifier(tf.keras.Model):
    """Thin wrapper so the HF model plays nicely with model.fit()/callbacks."""

    def __init__(self, hf_model, **kwargs):
        super().__init__(**kwargs)
        self.hf_model = hf_model

    def call(self, inputs, training=False):
        outputs = self.hf_model(pixel_values=inputs, training=training)
        return outputs.logits


model = ViTClassifier(hf_vit, name=MODEL_NAME)
model.build(input_shape=(None, 3, IMG_SIZE, IMG_SIZE))

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

model.summary()

# ============================================================
# CALLBACKS & TRAIN
# ============================================================
# NOTE: HF models nested inside a custom tf.keras.Model subclass don't serialize
# cleanly with model.save()/.keras format, so we checkpoint WEIGHTS ONLY.
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=outp("model.weights.h5"),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1),
]

train_start = time.time()

history = model.fit(
    train_dataset_prefetched,
    validation_data=validation_dataset_prefetched,
    epochs=EPOCHS,
    callbacks=callbacks,
)

total_training_time = time.time() - train_start

# Reload the exact best-epoch weights that were saved to disk
model.load_weights(outp("model.weights.h5"))

best_epoch = int(np.argmax(history.history["val_accuracy"])) + 1
final_val_accuracy = history.history["val_accuracy"][-1]
final_val_loss = history.history["val_loss"][-1]

print(f"\nTraining time: {total_training_time:.2f}s | Best epoch: {best_epoch}")

# ============================================================
# SAVE HISTORY & TRAINING CURVES
# ============================================================
with open(outp("history.pkl"), "wb") as f:
    pickle.dump(history.history, f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"], label="Training Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[0].set_title(f"{MODEL_NAME} - Accuracy")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history["loss"], label="Training Loss")
axes[1].plot(history.history["val_loss"], label="Validation Loss")
axes[1].set_title(f"{MODEL_NAME} - Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig(outp("accuracy_loss_curve.png"), dpi=300)
plt.show()

# ============================================================
# PREDICTIONS ON VALIDATION SET (+ timing)
# ============================================================
y_true = []
for _, lbls in validation_dataset:
    y_true.extend(lbls.numpy())
y_true = np.array(y_true)

pred_start = time.time()
y_logits = model.predict(validation_dataset, verbose=1)
total_prediction_time = time.time() - pred_start

y_prob = tf.nn.softmax(y_logits, axis=1).numpy()   # ViT head outputs logits, not softmax
y_pred = np.argmax(y_prob, axis=1)
avg_prediction_time = total_prediction_time / len(y_true)

print(f"Total prediction time: {total_prediction_time:.4f}s | Avg/image: {avg_prediction_time:.6f}s")

# ============================================================
# PERFORMANCE METRICS
# ============================================================
accuracy = accuracy_score(y_true, y_pred)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=class_indices, average="macro", zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=class_indices, average="weighted", zero_division=0
)

kappa = cohen_kappa_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)

metrics_dict = {
    "Accuracy": accuracy,
    "Precision_Macro": precision_macro,
    "Precision_Weighted": precision_weighted,
    "Recall_Macro": recall_macro,
    "Recall_Weighted": recall_weighted,
    "F1_Macro": f1_macro,
    "F1_Weighted": f1_weighted,
    "Cohens_Kappa": kappa,
    "MCC": mcc,
}

pd.DataFrame([metrics_dict]).to_csv(outp("metrics.csv"), index=False)
print(metrics_dict)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================
report = classification_report(
    y_true, y_pred, labels=class_indices, target_names=class_names,
    digits=4, zero_division=0,
)
with open(outp("classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)

print(report)

# ============================================================
# CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(y_true, y_pred, labels=class_indices)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label"); plt.ylabel("True Label")
plt.title(f"{MODEL_NAME} - Confusion Matrix")
plt.tight_layout()
plt.savefig(outp("confusion_matrix.png"), dpi=300)
plt.show()

print(cm)

# ============================================================
# PREDICTIONS & PROBABILITIES CSVs
# ============================================================
pred_df = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "true_label": [class_names[i] for i in y_true],
    "predicted_label": [class_names[i] for i in y_pred],
})
pred_df.to_csv(outp("predictions.csv"), index=False)

prob_df = pd.DataFrame(y_prob, columns=[f"prob_{c}" for c in class_names])
prob_df.insert(0, "y_true", y_true)
prob_df.to_csv(outp("prediction_probabilities.csv"), index=False)

# ============================================================
# MODEL SUMMARY
# ============================================================
trainable_params = int(sum(np.prod(v.shape) for v in model.trainable_weights))
non_trainable_params = int(sum(np.prod(v.shape) for v in model.non_trainable_weights))
total_params = trainable_params + non_trainable_params
model_size_mb = os.path.getsize(outp("model.weights.h5")) / (1024 ** 2)

with open(outp("model_summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))
    f.write("\n===================================\n")
    f.write(f"Total Parameters         : {total_params:,}\n")
    f.write(f"Trainable Parameters     : {trainable_params:,}\n")
    f.write(f"Non-trainable Parameters : {non_trainable_params:,}\n")
    f.write(f"Model Size (MB)          : {model_size_mb:.2f}\n")

# ============================================================
# EVALUATION RESULTS (final combined report)
# ============================================================
with open(outp("evaluation_results.txt"), "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write(f"{MODEL_NAME} — EVALUATION RESULTS\n")
    f.write("=" * 60 + "\n\n")

    f.write("--- Training Information ---\n")
    f.write(f"Total Training Time (s)  : {total_training_time:.2f}\n")
    f.write(f"Best Epoch               : {best_epoch}\n")
    f.write(f"Final Validation Accuracy: {final_val_accuracy:.4f}\n")
    f.write(f"Final Validation Loss    : {final_val_loss:.4f}\n\n")

    f.write("--- Prediction Timing ---\n")
    f.write(f"Total Prediction Time (s)     : {total_prediction_time:.4f}\n")
    f.write(f"Avg Prediction Time/Image (s) : {avg_prediction_time:.6f}\n\n")

    f.write("--- Performance Metrics ---\n")
    for k, v in metrics_dict.items():
        f.write(f"{k:<20}: {v:.4f}\n")

    f.write("\n--- Confusion Matrix (rows=true, cols=pred) ---\n")
    f.write("Classes: " + ", ".join(class_names) + "\n")
    f.write(np.array2string(cm) + "\n\n")

    f.write("--- Model Summary ---\n")
    f.write(f"Total Parameters         : {total_params:,}\n")
    f.write(f"Trainable Parameters     : {trainable_params:,}\n")
    f.write(f"Non-trainable Parameters : {non_trainable_params:,}\n")
    f.write(f"Model Size (MB)          : {model_size_mb:.2f}\n")

print(f"\nAll outputs saved to: {os.path.abspath(OUTPUT_DIR)}")
print("Training complete.")

ModuleNotFoundError: No module named 'numpy'

: 